<a href="https://colab.research.google.com/github/AtacZy/Projects/blob/main/%D0%9E%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D0%BB%D0%BE%D0%B3%D0%BE%D0%B2_%D0%BE_%D0%BF%D0%BE%D0%B4%D0%BF%D0%B8%D1%81%D0%BA%D0%B0%D1%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Обработка логов о подписках**

---

## **Цель:** Выполнить расчёты по трём задачам

---

## **Задачи**:
1. Написать функцию count_success_and_failure которая принимает на вход путь к файлу с логами и подсчитывает количество успешных продлений и ошибок при списании. Функция должна вернуть кортеж из двух значений: количества успешных попыток и неуспешных.
2. Напишите функцию auto_renewal_sub, которая принимает на вход путь к файлу с логами и обрабатывает количество клиентов с активированной функцией автопродления подписки. Рассмотрите изменение этого показателя в динамике путем расчета сглаженных значений двумя способами: методом скользящего среднего и методом медианного сглаживания.
3. Напишите функцию sub_renewal_by_day, которая принимает на вход путь к файлу с логами и анализирует взаимосвязь дня продления подписки и количества продлений в этот день. Функция должна записать в файл weekdays.txt аналитическую записку.

---

## **Автор**: AtacZy



## **Выводы по анализу продлений подписок**

---

### **№1. Общая статистика**

| Показатель | Количество | Процент |
|------------|-----------:|--------:|
| ✅ Успешные продления | 1 034 | 84,8% |
| ❌ Неуспешные продления | 186 | 15,2% |
| **Всего попыток** | **1 220** | **100%** |

🔹 **Конверсия 85%** — более 8 из 10 попыток успешны.  
🔹 Это **хороший показатель** для подписочного бизнеса.  
🔹 Платёжная система и процессы продления работают **стабильно**.

---

### **№2. Динамика и точка перелома**

**Ключевое наблюдение:**  
После ~220 элемента в данных произошёл **резкий рост** популярности подписки.

| Показатель | До точки перелома | После точки перелома |
|------------|------------------|---------------------|
| Медиана | 0 | 1 |
| Характер | Нерегулярные продления | Регулярные продления |

**📌 Рекомендации:**

1. **🔍 Проанализировать точку перелома**  
   Выяснить, что именно вызвало рост: маркетинговая кампания, сезонность, изменение продукта или внешние факторы.

2. **💎 Внедрить программы лояльности**  
   Длительный период с медианой = 0 указывает на отток после первой подписки. Удержание существующих клиентов дешевле привлечения новых.

3. **📅 Учитывать сезонность**  
   Если скачок связан с конкретными датами (праздники, акции), планировать маркетинговые активности заранее.

4. **🧪 Провести A/B тестирование**  
   Попробовать воспроизвести условия точки перелома в контролируемом эксперименте.

---

### **№3. Распределение по дням недели**

| День недели | Продлений | Относительно среднего |
|-------------|----------:|:---------------------:|
| Понедельник | 136 | ▾ -8% |
| Вторник | 144 | ▾ -2% |
| **Среда** | **162** | ▴ +10% |
| **Четверг** | **169** | ▴ +14% |
| Пятница | 145 | ▾ -2% |
| Суббота | 135 | ▾ -9% |
| Воскресенье | 143 | ▾ -3% |
| **Среднее** | **~148** | — |

#####Импортируйте данные

In [59]:
!wget https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log

--2026-03-07 13:21:09--  https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.github.com (gist.github.com)... 140.82.114.4
Connecting to gist.github.com (gist.github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log [following]
--2026-03-07 13:21:09--  https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 459418 (449K) [text/plain]
Saving to: ‘auto_purchase.log.1’

auto_purchase.log.1 100%[===================>] 448.65K  --.-KB/s    in 0.04s   

2026-03-07 13:21:09 (12

In [ ]:
with open('auto_purchase.log', 'r') as f:
    lines = f.readlines()

for line in lines[400:1000]:
    print(line)

### Задача 1
  
* Написать функцию count_success_and_failure которая принимает на вход путь к файлу с логами и подсчитывает количество успешных продлений и ошибок при списании. Функция должна вернуть кортеж из двух значений: количества успешных попыток и неуспешных.





In [ ]:
file_path = '/content/auto_purchase.log'

##### Решение:

In [ ]:
def count_success_and_failure(file_path):

# Создадим счётчики всех попыток обновления подписок и неуспешных
  succesful = 0
  failed = 0
# Читаем файл по нужным строкам и обновляем счётчики
  with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
      if 'Обновляем подписку пользователю id:' in line:
        succesful += 1
      elif 'ошибка при списании' in line:
        failed += 1
# Вычитаем из общего количества обновлений подписок ошибочные подписки
  result = succesful - failed
  return result, failed

count_success_and_failure(file_path)

(1034, 186)

In [ ]:
# Проверим решение
res = count_success_and_failure('auto_purchase.log')

try:
    assert res == (1034, 186)
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### Задача 2

Напишите функцию `auto_renewal_sub`, которая принимает на вход путь к файлу с логами и обрабатывает количество клиентов с активированной функцией автопродления подписки. Рассмотрите изменение этого показателя в динамике путем расчета сглаженных значений двумя способами: методом скользящего среднего и методом медианного сглаживания.

Примеры особенностей обработки:
- Под сглаживанием понимается использование всех предыдущих значений, включая текущее, но исключая будущие значения.
- Если за один день зарегистрировано несколько записей об активации автопродления, берется максимальное значение числа клиентов среди зафиксированных за этот день.

Результатом работы функции должно стать сохранение двух списков сглаженных значений в файл `auto_renewal_sub.txt`, при этом перед каждым списком указывается соответствующий заголовок:

##### Логика 2-ой задачи:
1. Импортируем необходимые библиотеки
2. Читаем файл и трансформируем в словарь с нужными элементами
3. Выбираем по нужной строке необходимые элементы и извлекаем оттуда количество подписок на каждый день в словарь
4. Делаем проверку на исключение
5. Считаем скользящее срденее для каждого значения и медиану аналогично
6. Записываем в файл с нужным наименованием по условию

##### Решение:

In [60]:
from datetime import datetime
import statistics

def auto_renewal_sub(file_path):
  # Читаем файл и трансформируем в словарь с нужными элементами
  with open(file_path, 'r') as file:
    logs = []
    for line in file:
      logs.append([part.strip() for part in line.split('|')])

  data = [{'timestamp': datetime.strptime(log[1], '%Y-%m-%d %H:%M:%S,%f'),
          'message': log[4].replace('[demon] ', '').strip(),
          'system': log[0]} for log in logs]

  # Выбираем по нужной строке необходимые элементы и извлекаем оттуда количество подписок на каждый день в словарь
  result = {}
  for item in data:
    date = item['timestamp'].strftime('%Y-%m-%d')
    if 'количество людей с автопродлением подписки' in item['message']:
      num = int(item['message'].split(': ')[-1])
      if date in result:
        result[date] = max(result[date], num)
      else:
        result[date] = num

  values = list(result.values())

  def avg_mean_median(values):
    # Делаем проверку на исключение
    if len(values) < 1:
      raise ValueError('Размер списка должен быть больше 1')
    # Считаем скользящее срденее для каждого значения и медиану аналогично
    mean = list(map(lambda i: round(sum(values[:i+1])/len(values[:i+1]), 2), range(len(values))))
    med = [int(statistics.median(values[:i+1])) for i in range(len(values))]
    return (mean, med)

  mean, med = avg_mean_median(values)

  with open('auto_renewal_sub.txt', 'w') as f:
    f.write(f'Среднее: {mean}\nМедиана: {med}')

auto_renewal_sub('auto_purchase.log')

In [ ]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt

import pandas as pd

user_answer = pd.read_csv('auto_renewal_sub.txt')
correct_answer = pd.read_csv('cor_auto_renewal.txt')

In [62]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!


### **Задача 3**

Напишите функцию `sub_renewal_by_day`, которая принимает на вход путь к файлу с логами и анализирует взаимосвязь дня продления подписки и количества продлений в этот день. Функция должна записать в файл `weekdays.txt` аналитическую записку в формате:

**`Количество обновлений подписки по дням недели:`**

**`Понедельник: 6`**

**`Вторник: 7`**

**`Среда: 8`**

**`...`**

*Примечание:* учитывайте в этой задаче все продления подписок, в том числе дублирующиеся.

##### **Логика:**
1. Отфильтровать данные по успешным продлениям в отдельный список
2. Перевести даты в номера недель и посчитать с помощью Counter
3. Записать в файл по шаблону


In [85]:
from collections import Counter

with open(file_path, 'r') as file:
  logs = []
  for line in file:
    logs.append([part.strip() for part in line.split('|')])

data = [{'timestamp': datetime.strptime(log[1], '%Y-%m-%d %H:%M:%S,%f'),
          'message': log[4].replace('[demon] ', '').strip(),
          'system': log[0]} for log in logs]
# Фильтруем успешные продления
success_count = [data[i] for i in range(len(data)) if 'Обновляем подписку пользователю id:' in data[i]['message']
                 and (i == len(data)-1 or 'ошибка при списании' not in data[i+1]['message'])]
# Создаём словарь с наименованиями дней недели
weekdays_ru = {
    0: 'Понедельник',
    1: 'Вторник',
    2: 'Среда',
    3: 'Четверг',
    4: 'Пятница',
    5: 'Суббота',
    6: 'Воскресенье'
}
# Считаем с помощью Counter
day_counts = Counter([weekdays_ru[item['timestamp'].weekday()] for item in success_count])
# Берем только значения
weekname = weekdays_ru.values()

# Записываем в файл:
with open('weekdays.txt', 'w') as f:
  print('Количество обновлений подписки по дням недели:', file=f)
  for day_name in weekname:
    count = day_counts.get(day_name, 0)
    f.write(f'{day_name}: {count}\n')

In [ ]:
!wget https://gist.github.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt

import pandas as pd

user_answer = pd.read_csv('weekdays.txt')
correct_answer = pd.read_csv('cor_weekdays.txt')

In [87]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!
